# Predicting Negative Customer Reviews in Brazilian E-Commerce

This notebook gives a compact view of the full analysis. The main question is how early negative-review risk can be identified and how much the signal improves once delivery information becomes available.

**Target:** `review_bad = 1` for review scores 1–2 and `0` for scores 3–5.  
**Final sample:** 95,824 delivered orders; 12.8% have negative reviews.

## 1. Model comparison

Both models use the same frozen test set. The placement model uses only information available when the order is created. The delivery model adds observed fulfillment information.

In [1]:
import pandas as pd

comparison = pd.read_csv('../outputs/model_metrics/placement_vs_delivery.csv')
comparison[['stage','model','precision','recall','f1','roc_auc','pr_auc']].round(3)

,stage,model,precision,recall,f1,roc_auc,pr_auc
0,At Placement,LightGBM,0.219,0.473,0.299,0.658,0.233
1,At Delivery,CatBoost,0.565,0.431,0.489,0.775,0.496


The delivery model has much better precision and ranking quality. The placement model keeps more recall, which makes it more suitable as an early screening score.

![Model comparison](../assets/model_comparison.svg)

## 2. Order personas

K-Means provides a descriptive view of recurring order patterns. The clusters are fit without review outcome, then profiled using review and delivery results.

In [2]:
clusters = pd.read_csv('../outputs/interpretation/cluster_profile.csv')
clusters[['persona','size','share','bad_review_rate','late_rate','avg_delivery_time_days','avg_n_items']].round(3)

,persona,size,share,bad_review_rate,late_rate,avg_delivery_time_days,avg_n_items
0,Long Wait,16705,0.174,0.291,0.395,25.289,1.026
1,Multi-Item Basket,8306,0.087,0.259,0.032,11.340,2.482
2,Big-Ticket Planner,26692,0.279,0.082,0.016,11.783,1.002
3,Quick Small Buy,44121,0.460,0.069,0.008,8.347,1.017


The Long Wait and Multi-Item Basket profiles have the highest negative-review rates, but for different reasons. Long Wait is closely tied to poor delivery performance, while Multi-Item Basket suggests that order complexity can also matter.

![Persona risk](../assets/persona_risk.svg)

## 3. Delivery-model drivers

In [3]:
importance = pd.read_csv('../outputs/interpretation/catboost_feature_importance.csv')
importance.head(12).round(3)

,feature,importance
0,late_days,16.760
1,delivery_vs_estimate_days,15.422
2,n_items,11.205
3,delivery_time_days,7.088
4,primary_product_category,4.689
5,primary_seller_city,4.137
6,carrier_pickup_delay_days,3.709
7,customer_city,2.871
8,n_distinct_sellers,2.797
9,promised_delivery_days,2.350


The strongest features are related to lateness, performance against the estimated delivery date, number of items and delivery duration. Feature importance is used here as a predictive diagnostic, not as a causal estimate.

![Feature importance](../assets/feature_importance.svg)

## 4. Risk concentration

A useful score should do more than cross a classification threshold. For a service team with limited capacity, the ranking itself matters.

In [4]:
lift = pd.read_csv('../outputs/model_metrics/lift_capture.csv')
lift.round(3)

,stage,top_share,orders,negative_reviews_captured,segment_negative_review_rate,lift_vs_average
0,At Placement,0.05,959,0.135,0.346,2.704
1,At Placement,0.10,1917,0.231,0.296,2.310
2,At Placement,0.20,3833,0.377,0.242,1.887
3,At Delivery,0.05,959,0.298,0.763,5.961
4,At Delivery,0.10,1917,0.436,0.559,4.363
5,At Delivery,0.20,3833,0.579,0.371,2.897


On the held-out test set, the top 10% of CatBoost risk scores contain about **43.6% of all negative reviews**. The negative-review rate inside that group is about **55.9%**, roughly **4.4× the overall rate**.

This makes the delivery score a reasonable basis for a targeted service-recovery queue rather than a broad outreach list.

## 5. Clustering robustness

DBSCAN provides a structurally different check on the same standardized clustering variables. With `eps=1.0` and `min_samples=20`, about **3.6%** of orders are labelled as noise. Agreement with K-Means is low (ARI ≈ **0.159**), but the noise group has a negative-review rate of about **32.9%**.

I keep K-Means for the main personas because it produces four stable, readable segments. DBSCAN is more useful here as an outlier signal than as the primary segmentation.


## 6. Capacity and cost sensitivity

The model is also evaluated as an operating queue. Reviewing the highest-risk **5%** of delivery scores captures about **29.8%** of negative reviews; expanding the queue to **10%** raises capture to about **43.6%**.

I use a normalized false-positive / false-negative cost model instead of inventing monetary assumptions that are not in the dataset. At the validation-selected thresholds, the placement model produces more false positives but slightly fewer false negatives. The break-even false-negative-to-false-positive cost ratio is about **32.7:1**. Below that ratio, the delivery-stage policy is cheaper; above it, the earlier placement-stage policy becomes cheaper. This is a sensitivity result, not an estimate of Olist's actual economics.


## 7. Next step

Notebook 06 combines the order-level data, personas and saved model probabilities into the single table used by Power BI. The final report will focus on executive risk monitoring, drivers and a customer-recovery queue.
